In [1]:
# Load the uploaded Excel macro-enabled workbook and inspect available sheets
import pandas as pd

file_path = "Town_Ward.xlsm"

# Read sheet names first (fast, no full load)
xl = pd.ExcelFile(file_path)
sheet_names = xl.sheet_names
print(sheet_names)

['01_SR', 'SAD_SAZ', '02_District', '03_Township', '04_Town', '_05_Ward', '_06-VillageTract', 'Metadata']


In [2]:
# Load key sheets into dataframes (kept in memory) and preview heads
import pandas as pd

file_path = "Town_Ward.xlsm"

sheet_to_df = {}
for sheet in ['01_SR', '02_District', '03_Township', '04_Town', '_05_Ward', '_06-VillageTract']:
    sheet_to_df[sheet] = pd.read_excel(file_path, sheet_name=sheet)

for sheet, df in sheet_to_df.items():
    print(sheet)
    print(df.head())
    print("rows")
    print(len(df))
    print("cols")
    print(len(df.columns))
    print("-")

01_SR
  SR_Pcode SR_Name_Eng             SR_Name_MMR MYAINFO_SD_ID Source  \
0   MMR001      Kachin            ကချင်ပြည်နယ်     ASIMMR001    GAD   
1   MMR002       Kayah             ကယားပြည်နယ်     ASIMMR002    GAD   
2   MMR003       Kayin             ကရင်ပြည်နယ်     ASIMMR003    GAD   
3   MMR004        Chin            ချင်းပြည်နယ်     ASIMMR004    GAD   
4   MMR005     Sagaing  စစ်ကိုင်းတိုင်းဒေသကြီး     ASIMMR005    GAD   

   Start_Date  Modified_End_Date  Notification  Notification_modified  \
0         NaN                NaN           NaN                    NaN   
1         NaN                NaN           NaN                    NaN   
2         NaN                NaN           NaN                    NaN   
3         NaN                NaN           NaN                    NaN   
4         NaN                NaN           NaN                    NaN   

  GAD_State/Region_Status  Field_State/Region_Status  \
0                  Active                        NaN   
1               

In [3]:
# Clean the township/town/ward/village-tract sheets (they have multi-row headers) and standardize key columns
import pandas as pd
import numpy as np

# Use already-loaded sheet_to_df from memory

def clean_multiline_header(df_raw):
    df = df_raw.copy()
    # Find first row that looks like real data (has a pcode pattern)
    pcode_like = df.apply(lambda col: col.astype(str).str.contains(r"MMR", na=False)).any(axis=1)
    first_data_idx = int(np.argmax(pcode_like.values)) if pcode_like.any() else 0
    df = df.iloc[first_data_idx:].reset_index(drop=True)
    # Use the row as header if current columns are Unnamed
    if any(str(c).startswith("Unnamed") for c in df.columns):
        new_cols = df.iloc[0].astype(str).tolist()
        df = df.iloc[1:].reset_index(drop=True)
        df.columns = new_cols
    # Drop fully empty columns
    df = df.dropna(axis=1, how="all")
    # Trim column names
    df.columns = [str(c).strip().replace("\
", " ") for c in df.columns]
    return df

sr_df = sheet_to_df['01_SR'].copy()
district_df = sheet_to_df['02_District'].copy()
township_df = clean_multiline_header(sheet_to_df['03_Township'])
town_df = clean_multiline_header(sheet_to_df['04_Town'])
ward_df = clean_multiline_header(sheet_to_df['_05_Ward'])
vt_df = clean_multiline_header(sheet_to_df['_06-VillageTract'])

print(sr_df.head())
print(district_df.head())
print(township_df.head())
print(ward_df.head())
print(vt_df.head())

  SR_Pcode SR_Name_Eng             SR_Name_MMR MYAINFO_SD_ID Source  \
0   MMR001      Kachin            ကချင်ပြည်နယ်     ASIMMR001    GAD   
1   MMR002       Kayah             ကယားပြည်နယ်     ASIMMR002    GAD   
2   MMR003       Kayin             ကရင်ပြည်နယ်     ASIMMR003    GAD   
3   MMR004        Chin            ချင်းပြည်နယ်     ASIMMR004    GAD   
4   MMR005     Sagaing  စစ်ကိုင်းတိုင်းဒေသကြီး     ASIMMR005    GAD   

   Start_Date  Modified_End_Date  Notification  Notification_modified  \
0         NaN                NaN           NaN                    NaN   
1         NaN                NaN           NaN                    NaN   
2         NaN                NaN           NaN                    NaN   
3         NaN                NaN           NaN                    NaN   
4         NaN                NaN           NaN                    NaN   

  GAD_State/Region_Status  Field_State/Region_Status  \
0                  Active                        NaN   
1                  Act

In [4]:
# Further clean: drop instructional rows, remove all-nan columns, keep core columns, and compute summary metrics
import pandas as pd
import numpy as np


def drop_instruction_rows(df):
    df2 = df.copy()
    first_col = df2.columns[0]
    mask_bad = df2[first_col].astype(str).str.contains("Enter seach text", na=False) | (df2[first_col].astype(str).str.strip() == "None")
    df2 = df2[~mask_bad].copy()
    df2 = df2.replace({"None": np.nan})
    df2 = df2.dropna(how="all")
    return df2.reset_index(drop=True)

# Apply to cleaned hierarchy sheets
if 'township_df' in globals():
    township_df2 = drop_instruction_rows(township_df)
    ward_df2 = drop_instruction_rows(ward_df)
    vt_df2 = drop_instruction_rows(vt_df)
    town_df2 = drop_instruction_rows(town_df)
else:
    township_df2 = township_df
    ward_df2 = ward_df
    vt_df2 = vt_df
    town_df2 = town_df

# Drop columns named nan, nan.1, ...
def drop_nan_named_cols(df):
    good_cols = [c for c in df.columns if not str(c).lower().startswith('nan')]
    return df[good_cols].copy()

township_df2 = drop_nan_named_cols(township_df2)
town_df2 = drop_nan_named_cols(town_df2)
ward_df2 = drop_nan_named_cols(ward_df2)
vt_df2 = drop_nan_named_cols(vt_df2)

print(township_df2.head())
print(ward_df2.head())
print(vt_df2.head())

# Summary counts
sr_count = sr_df['SR_Pcode'].nunique(dropna=True)
district_count = district_df['District_Pcode'].nunique(dropna=True)
township_count = township_df2['Tsp_Pcode'].nunique(dropna=True) if 'Tsp_Pcode' in township_df2.columns else len(township_df2)
town_count = town_df2['Town_Pcode'].nunique(dropna=True) if 'Town_Pcode' in town_df2.columns else len(town_df2)
ward_count = ward_df2['Ward_Pcode'].nunique(dropna=True) if 'Ward_Pcode' in ward_df2.columns else len(ward_df2)
vt_count = vt_df2['VT_Pcode'].nunique(dropna=True) if 'VT_Pcode' in vt_df2.columns else len(vt_df2)

print(sr_count)
print(district_count)
print(township_count)
print(town_count)
print(ward_count)
print(vt_count)

  S R _ P c o d e  S R _ N a m e _ E n g   \
0         SR_Pcode            SR_Name_Eng   
1           MMR017             Ayeyarwady   
2           MMR017             Ayeyarwady   
3           MMR017             Ayeyarwady   
4           MMR017             Ayeyarwady   

  D i s t r i c t / S A Z _ P c o d e   \
0                   District/SAZ_Pcode   
1                           MMR017D006   
2                           MMR017D005   
3                           MMR017D006   
4                           MMR017D003   

  D i s t r i c t / S A Z _ N a m e _ E n g  T s p _ P c o d e   \
0                      District/SAZ_Name_Eng          Tsp_Pcode   
1                                     Pyapon          MMR017024   
2                                     Maubin          MMR017022   
3                                     Pyapon          MMR017026   
4                                  Myaungmya          MMR017015   

  T o w n s h i p _ N a m e _ E n g  T o w n s h i p _ N a m e _ M M R   

In [5]:
# Finalize cleaning by removing duplicated header rows that got embedded as first data row
import pandas as pd

def drop_header_repeat(df):
    df2 = df.copy()
    first_col = df2.columns[0]
    df2 = df2[df2[first_col].astype(str).str.strip() != str(first_col).strip()].copy()
    return df2.reset_index(drop=True)

township_df3 = drop_header_repeat(township_df2)
town_df3 = drop_header_repeat(town_df2)
ward_df3 = drop_header_repeat(ward_df2)
vt_df3 = drop_header_repeat(vt_df2)

print(township_df3.head())
print(ward_df3.head())
print(vt_df3.head())

  S R _ P c o d e  S R _ N a m e _ E n g   \
0         SR_Pcode            SR_Name_Eng   
1           MMR017             Ayeyarwady   
2           MMR017             Ayeyarwady   
3           MMR017             Ayeyarwady   
4           MMR017             Ayeyarwady   

  D i s t r i c t / S A Z _ P c o d e   \
0                   District/SAZ_Pcode   
1                           MMR017D006   
2                           MMR017D005   
3                           MMR017D006   
4                           MMR017D003   

  D i s t r i c t / S A Z _ N a m e _ E n g  T s p _ P c o d e   \
0                      District/SAZ_Name_Eng          Tsp_Pcode   
1                                     Pyapon          MMR017024   
2                                     Maubin          MMR017022   
3                                     Pyapon          MMR017026   
4                                  Myaungmya          MMR017015   

  T o w n s h i p _ N a m e _ E n g  T o w n s h i p _ N a m e _ M M R   

In [ ]:
    # township_df3.shape
    # town_df3
    # ward_df3
    # vt_df3

(359, 128)